# **Oxford Flowers Clustering Notebook**

In this notebook, we:
1. Load the validation and test splits of the Oxford Flower Dataset, merging them.
2. Compute the CLIP embeddings with the `ClipEmbedder`.
3. Cluster these images into 102 clusters (the number of classes).
4. Compute ARI, NMI, and interpret the results.

---

## **1. Setup and Load Data**

In [ ]:
import numpy as np

from pyvisim.distance import cosine_similarity
from pyvisim.datasets import OxfordFlowerDataset
from pyvisim.neural_networks import ClipEmbedder

from tutorial_utils import cluster_images_and_generate_statistics, plot_and_save_heatmap

# Load the validation and test datasets.

In [ ]:
dataset = OxfordFlowerDataset(purpose=["validation", "test"])
print("Number of images in the dataset:", len(dataset))

## **2. Compute CLIP embeddings**

#### Define the CLIP embedder

In [ ]:
clip_embedder = ClipEmbedder(variant="ViT-B/32", pretrained="openai")

#### Compute the CLIP embeddings for both the validation and test splits

In [ ]:
clip_embeddings = clip_embedder.embed(image for image, *_ in dataset)

## **3. Cluster into 102 Clusters and Compute ARI, NMI**

`102` is the number of classes in the Oxford Flowers dataset. We want to see how well the clustering algorithm can cluster the images into these classes.


In [ ]:
num_classes = 102
results = cluster_images_and_generate_statistics(
    features=clip_embeddings,  # The subset corresponding to val+test
    true_labels=np.array(dataset.labels),  # The true labels
    n_clusters=num_classes,
    method="spectral",
)

print(f"Clustering with Spectral Clustering into {num_classes} clusters:")
print("RI:", results["ri"])
print("ARI:", results["ari"])
print("NMI:", results["nmi"])

Now, instead of using the embeddings themselves, we will use the `similarity matrix` of the embeddings to cluster the images. So each row in this matrix will represent the similarity of an image to all other images in the dataset (hence, all diagonal elements will be 1).

In [ ]:
feature_map = cosine_similarity(clip_embeddings, clip_embeddings)
plot_and_save_heatmap(
    feature_map[:10, :10],
    title="Similarity Matrix of test dataset, first 10 images",
    x_label="Image Index",
    y_label="Image Index",
)

In [ ]:
results = cluster_images_and_generate_statistics(
    features=feature_map,  # The subset corresponding to val+test
    true_labels=np.array(dataset.labels),  # The true labels
    n_clusters=num_classes,
    method="spectral",
)
print(f"Clustering with Spectral Clustering into {num_classes} clusters:")
print("RI:", results["ri"])
print("ARI:", results["ari"])
print("NMI:", results["nmi"])

## **4. Conclusion**

We've demonstrated:
- How to cluster images directly on CLIP embeddings.
- How to compute ARI and NMI for objective evaluation.

For CLIP, clustering on the embeddings themselves performs significantly better than clustering on the similarity matrix.